In [ ]:
# SETUP 
import os
os.environ["WANDB_DISABLED"] = "true"  # Disable Weights and Biases logging
%pip install transformers datasets evaluate torch  # Install necessary packages

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, Trainer, TrainingArguments
from datasets import load_dataset
import evaluate
import time

In [ ]:
# LOAD DATA 
dataset = load_dataset(" ") # Fill in
train_data = dataset["train"]
test_data = dataset["test"]

In [ ]:
# TOKENIZER
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

In [ ]:
# MODEL
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

In [ ]:
# METRICS 
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


In [ ]:
# TRAINING 
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy="epoch",
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# TRAIN
print("Training model...")
start_time = time.time()
trainer.train()

if torch.cuda.is_available():
    print(f"GPU Memory After Training: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

training_time = time.time() - start_time
print(f"Training time: {training_time:.2f} seconds")

# ==== EVALUATE ====
print("Evaluating...")
start_time = time.time()
results = trainer.evaluate()

if torch.cuda.is_available():
    print(f"GPU Memory After Eval: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

inference_time = time.time() - start_time
print(f"Inference time: {inference_time:.2f} seconds")

print(results)